In [55]:
import os
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit import Chem, DataStructs
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

In [56]:
ROOT_DIR = os.path.join('/home', 'rlawlsgurjh', 'work', 'MPOCSR')

In [57]:
# TASK_NAME = 'non_aug'
TASK_NAME = 'ocsaug'

In [58]:
RESULTS_DIR = os.path.join(ROOT_DIR, 'model', 'MPViT-CBloss', 'output', 'MPViT_CBloss', TASK_NAME)

In [59]:
def tanimoto_similarity(smiles1, smiles2):
    try:
        # SMILES를 분자 객체로 변환
        mol1 = Chem.MolFromSmiles(smiles1)
        mol2 = Chem.MolFromSmiles(smiles2)
        
        # 두 분자 모두 유효한지 확인
        if mol1 is None or mol2 is None:
            return 0
        
        # 분자 지문 계산
        fp1 = Chem.RDKFingerprint(mol1)
        fp2 = Chem.RDKFingerprint(mol2)
        
        # Tanimoto similarity 계산
        tanimoto = DataStructs.FingerprintSimilarity(fp1, fp2)
        return tanimoto
    except:
        return 0

In [60]:
def compute_tanimoto_similarity(smiles_list, pred_smiles_list):
    return [tanimoto_similarity(g, p) for g, p in zip(smiles_list, pred_smiles_list)]

In [61]:
all_results = []
for filename in os.listdir(RESULTS_DIR):
    if filename.endswith('predictions.csv'):
        checkpoint_name = filename.replace('_predictions.csv', '')
        file_path = os.path.join(RESULTS_DIR, filename)
        
        # CSV 파일 읽기
        df = pd.read_csv(file_path)
        
        # Tanimoto Similarity 계산
        ts_scores = compute_tanimoto_similarity(df['true_smiles'], df['pred_smiles'])
        mean_ts = np.mean(ts_scores)
        
        df['TS'] = ts_scores
        
        # 결과 저장
        all_results.append({
            'checkpoint': checkpoint_name,
            'tanimoto_similarity': mean_ts
        })
        
        # 새 파일 이름 생성 (_predictions.csv -> _predictions_TS.csv)
        new_filename = filename.replace('.csv', '_TS.csv')
        new_file_path = os.path.join(RESULTS_DIR, new_filename)
        
        # 새 CSV 파일로 저장
        df.to_csv(new_file_path, index=False)
                
        print(f"체크포인트 {checkpoint_name}의 평균 Tanimoto Similarity: {mean_ts:.3f}")

# 모든 결과를 DataFrame으로 변환
results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(by='checkpoint', ascending=True)

print(results_df.head())

체크포인트 mpvit_transform_celoss_19의 평균 Tanimoto Similarity: 0.603
체크포인트 mpvit_transform_celoss_17의 평균 Tanimoto Similarity: 0.603
체크포인트 mpvit_transform_celoss_3의 평균 Tanimoto Similarity: 0.603
체크포인트 mpvit_transform_celoss_4의 평균 Tanimoto Similarity: 0.603
체크포인트 mpvit_transform_celoss_15의 평균 Tanimoto Similarity: 0.603
체크포인트 mpvit_transform_celoss_16의 평균 Tanimoto Similarity: 0.603
체크포인트 mpvit_transform_celoss_23의 평균 Tanimoto Similarity: 0.603
체크포인트 mpvit_transform_celoss_2의 평균 Tanimoto Similarity: 0.603
체크포인트 mpvit_transform_celoss_26의 평균 Tanimoto Similarity: 0.603
체크포인트 mpvit_transform_celoss_18의 평균 Tanimoto Similarity: 0.603
체크포인트 mpvit_transform_celoss_20의 평균 Tanimoto Similarity: 0.603
체크포인트 mpvit_transform_celoss_11의 평균 Tanimoto Similarity: 0.603
체크포인트 mpvit_transform_celoss_0의 평균 Tanimoto Similarity: 0.603
체크포인트 mpvit_transform_celoss_6의 평균 Tanimoto Similarity: 0.603
체크포인트 mpvit_transform_celoss_1의 평균 Tanimoto Similarity: 0.603
체크포인트 mpvit_transform_celoss_29의 평균 Tanimoto Similarity: 0.60

In [62]:
print(results_df)

                   checkpoint  tanimoto_similarity
12   mpvit_transform_celoss_0             0.603288
14   mpvit_transform_celoss_1             0.603288
16  mpvit_transform_celoss_10             0.603288
11  mpvit_transform_celoss_11             0.603288
22  mpvit_transform_celoss_12             0.603288
19  mpvit_transform_celoss_13             0.603288
21  mpvit_transform_celoss_14             0.603288
4   mpvit_transform_celoss_15             0.603288
5   mpvit_transform_celoss_16             0.603288
1   mpvit_transform_celoss_17             0.603288
9   mpvit_transform_celoss_18             0.603288
0   mpvit_transform_celoss_19             0.603288
7    mpvit_transform_celoss_2             0.603288
10  mpvit_transform_celoss_20             0.603288
18  mpvit_transform_celoss_21             0.603288
17  mpvit_transform_celoss_22             0.603288
6   mpvit_transform_celoss_23             0.603288
26  mpvit_transform_celoss_24             0.603288
23  mpvit_transform_celoss_25  

In [63]:
# pre_df = pd.read_csv(os.path.join(RESULTS_DIR, '..', 'default', 'mpvit_pre_predictions.csv'))

# ts_scores = compute_tanimoto_similarity(pre_df['true_smiles'], pre_df['pred_smiles'])
# mean_ts = np.mean(ts_scores)

# print(mean_ts)